# 02 — Fused Softmax

**Ladder:** Multi-pass naive → per-row fuse → single Triton kernel.

**Study tip.** First count HBM traffic (naive vs fused). Then implement stable softmax with $\max$ subtraction; padded lanes must not spoil $\max$/sum.

Official: [Fused Softmax](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

## 0. Definition and numerical stability

Fix a row $x \in \mathbb{R}^{N}$. The **softmax** is

$$
\mathrm{softmax}(x)_i
=
\frac{e^{x_i}}{\sum_{j=0}^{N-1} e^{x_j}},
\qquad i = 0,\ldots,N-1.
$$

**Proposition (shift invariance).** For any $c \in \mathbb{R}$,

$$
\mathrm{softmax}(x) = \mathrm{softmax}(x - c\mathbf{1}).
$$

*Proof.*

$$
\frac{e^{x_i - c}}{\sum_j e^{x_j - c}}
=
\frac{e^{-c} e^{x_i}}{e^{-c} \sum_j e^{x_j}}
=
\frac{e^{x_i}}{\sum_j e^{x_j}}.
$$

**Stable form.** Choose $c = m := \max_j x_j$. Then $x_i - m \le 0$, so $e^{x_i-m} \le 1$ and we avoid overflow:

$$
\mathrm{softmax}(x)_i
=
\frac{\exp(x_i - m)}{\sum_{j} \exp(x_j - m)}.
$$

**Probability simplex.** Writing $p = \mathrm{softmax}(x)$, we have $p_i \ge 0$ and $\sum_i p_i = 1$.

**Running example (padding).** $N=5$ → $B=\mathrm{next\_power\_of\_2}(5)=8$; lanes $j\ge 5$ load $-\infty$.

**Teaching / bench sizes.** Pseudo tests use $32\times 64$; official bench fixes $M=4096$ and sweeps column count $N$.

### Visual: theory board (method / logic)

![Whiteboard: multi-pass HBM softmax vs fused one-row kernel](docs/images/benchmark-softmax.png)

*Theory board — five HBM passes (~$8MNb$) vs fused row softmax (~$2MNb$ traffic).*

### Theory board

![Whiteboard: multi-pass HBM softmax vs fused one-row kernel](docs/images/whiteboard-theory.jpg)

*Theory board — five HBM passes (~$8MNb$) vs fused row softmax (~$2MNb$ traffic).*

### Pitfalls

- Softmax without subtracting max → overflow in exp.
- Multi-pass PyTorch looks simple but multiplies HBM traffic.
- BLOCK_SIZE must cover N (next_power_of_2); mask invalid lanes as -inf for max.

### Self-check

1. Derive traffic ~8MNb vs ~2MNb for naive vs fused.
2. Prove softmax(x)=softmax(x-c) for any scalar c.
3. When would fusion not help? (tiny rows already cache-resident)


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl
from triton.runtime import driver

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



def is_hip() -> bool:
    try:
        return triton.runtime.driver.active.get_current_target().backend == "hip"
    except Exception:
        return False


def is_cdna() -> bool:
    try:
        return is_hip() and triton.runtime.driver.active.get_current_target().arch in (
            "gfx940",
            "gfx941",
            "gfx942",
            "gfx90a",
        )
    except Exception:
        return False


## 1. Naive multi-pass (memory proof)

Implement softmax as five sequential tensor passes on a matrix $X \in \mathbb{R}^{M \times N}$ (row-wise):

| Pass | Map | HBM traffic (order) |
|---|---|---|
| 1 | $m_r = \max_j X_{rj}$ | read $X$ |
| 2 | $Z_{rj} = X_{rj} - m_r$ | read $X$, write $Z$ |
| 3 | $U_{rj} = e^{Z_{rj}}$ | read $Z$, write $U$ |
| 4 | $s_r = \sum_j U_{rj}$ | read $U$ |
| 5 | $Y_{rj} = U_{rj}/s_r$ | read $U$, write $Y$ |

**Traffic lower bound (sketch).** Ignoring tiny $M$-vectors, this is $\Theta(M N)$ bytes **several times**. With element size $b$,

$$
T_{\mathrm{naive}} \approx (8MN + 4M)\,b,
\qquad
T_{\mathrm{fused}} \approx 2MN\,b
\quad\Rightarrow\quad
\frac{T_{\mathrm{naive}}}{T_{\mathrm{fused}}} \xrightarrow{N\to\infty} 4.
$$

**Concrete check.** For $M=32$, $N=64$, $b=4$: fused traffic $\approx 16\,\mathrm{KiB}$ of element moves vs roughly $4\times$ that for the naive chain.


In [ ]:
def naive_softmax(x):
    """
    Numerically stable softmax as a sequence of full-tensor passes.
    (See markdown cells above for the math.)
    """
    # Pass 1: row-wise max (for numerical stability)
    x_max = x.max(dim=1)[0]
    # Pass 2: subtract max
    z = x - x_max[:, None]
    # Pass 3: exp
    numerator = torch.exp(z)
    # Pass 4: row-wise sum
    denominator = numerator.sum(dim=1)
    # Pass 5: normalize
    return numerator / denominator[:, None]


## 2. Per-row fuse (same math, one program owns a row)

For each row index $r$:

$$
\begin{aligned}
m_r &= \max_j X_{rj}, \\
U_{rj} &= \exp(X_{rj} - m_r), \\
s_r &= \sum_j U_{rj}, \\
Y_{rj} &= U_{rj} / s_r.
\end{aligned}
$$

**Claim.** This equals $\mathrm{softmax}$ applied to row $r$ (by §0).

Intermediates $m_r,U,s_r$ need not be full $M \times N$ global tensors — they can live in registers/SRAM for that row. Triton makes that concrete with one kernel launch and a persistent row schedule.

**Teaching shape.** $X\in\mathbb{R}^{32\times 64}$ (as in `test_pseudo_matches_torch`): each "program" conceptually walks one row end-to-end before moving on.


In [ ]:
def row_softmax_pseudo(x):
    """
    Same math as naive_softmax, but structured like the Triton kernel:
    (See markdown cells above for the math.)
    """
    M, N = x.shape
    y = torch.empty_like(x)

    for row_idx in range(M):
        row = x[row_idx]  # one logical load of the row
        row_max = row.max()
        numer = torch.exp(row - row_max)
        denom = numer.sum()
        y[row_idx] = numer / denom  # one logical store of the row

    return y


## 3. Triton fused kernel

Program $\mathrm{pid}$ walks rows $\mathrm{pid},\;\mathrm{pid}+P,\;\mathrm{pid}+2P,\ldots$ (`tl.num_programs`). For each row, the stable formula of §0 is evaluated entirely in-register after one masked `tl.load`.

**Block size / padding.**

$$
B = 2^{\lceil\log_2 N\rceil},
\qquad
\mathrm{row}_j =
\begin{cases}
X_{i,j} & j < N \\
-\infty & N \le j < B
\end{cases}
$$

so $\mathtt{tl.max}$ ignores padding and $\exp(-\infty)=0$ does not pollute the sum.

**Complexity.** One read of $X$, one write of $Y$ $\Rightarrow$ bandwidth

$$
\mathrm{GB/s} \approx \frac{2 M N b}{t \cdot 10^{9}}.
$$

| Stage | Program owns | Memory pattern |
|---|---|---|
| `naive_softmax` | whole tensor (many passes) | ~$8MN$ element trips |
| `row_softmax_pseudo` | one row at a time | intermediates local |
| `softmax_kernel` | strided set of rows | one load + one store per row |


In [ ]:
@triton.jit
def softmax_kernel(
    output_ptr,
    input_ptr,
    input_row_stride,
    output_row_stride,
    n_rows,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
    num_stages: tl.constexpr,
):
    # Persistent programs: each pid walks a strided set of rows
    # (same "one row at a time" idea as row_softmax_pseudo).
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx * input_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        input_ptrs = row_start_ptr + col_offsets
        mask = col_offsets < n_cols
        # Pad OOB with -inf so max/exp ignore them
        row = tl.load(input_ptrs, mask=mask, other=-float("inf"))
        row_minus_max = row - tl.max(row, axis=0)
        numerator = tl.exp(row_minus_max)
        denominator = tl.sum(numerator, axis=0)
        softmax_output = numerator / denominator
        output_row_start_ptr = output_ptr + row_idx * output_row_stride
        output_ptrs = output_row_start_ptr + col_offsets
        tl.store(output_ptrs, softmax_output, mask=mask)


def _device_props():
    """Lazy so importing this module on CPU (no Triton driver) still works for pseudo tests."""
    properties = driver.active.utils.get_device_properties(DEVICE.index)
    return properties


def softmax(x):
    properties = _device_props()
    NUM_SM = properties["multiprocessor_count"]
    NUM_REGS = properties["max_num_regs"]
    SIZE_SMEM = properties["max_shared_mem"]
    WARP_SIZE = properties["warpSize"]

    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    num_warps = 8
    num_stages = 4 if SIZE_SMEM > 200000 else 2
    y = torch.empty_like(x)
    # Warmup to read register / smem usage, then pick occupancy-aware grid
    kernel = softmax_kernel.warmup(
        y,
        x,
        x.stride(0),
        y.stride(0),
        n_rows,
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
        num_stages=num_stages,
        num_warps=num_warps,
        grid=(1,),
    )
    kernel._init_handles()
    n_regs = kernel.n_regs
    size_smem = kernel.metadata.shared
    if is_hip():
        num_gprs = NUM_REGS * 2 if is_cdna() else NUM_REGS
        max_num_threads = properties["max_threads_per_sm"]
        max_num_waves = max_num_threads // WARP_SIZE
        occupancy = min(num_gprs // WARP_SIZE // n_regs, max_num_waves) // num_warps
    else:
        occupancy = NUM_REGS // (n_regs * WARP_SIZE * num_warps)
    occupancy = min(occupancy, SIZE_SMEM // size_smem)
    num_programs = min(NUM_SM * occupancy, n_rows)
    kernel[(num_programs, 1, 1)](
        y, x, x.stride(0), y.stride(0), n_rows, n_cols, BLOCK_SIZE, num_stages
    )
    return y


In [ ]:
def test_pseudo_matches_torch() -> None:
    """CPU-runnable: naive and per-row pseudo match torch.softmax."""
    torch.manual_seed(0)
    x = torch.randn(32, 64)
    ref = torch.softmax(x, dim=1)
    assert torch.allclose(naive_softmax(x), ref, atol=1e-5, rtol=1e-5)
    assert torch.allclose(row_softmax_pseudo(x), ref, atol=1e-5, rtol=1e-5)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(1823, 781, device=DEVICE)
    y_triton = softmax(x)
    y_torch = torch.softmax(x, axis=1)
    assert torch.allclose(y_triton, y_torch)
    print("correctness OK")


In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["N"],
        x_vals=[128 * i for i in range(2, 40)],
        line_arg="provider",
        line_vals=["triton", "torch", "naive_softmax"],
        line_names=["Triton", "Torch", "Naive Softmax"],
        styles=[("blue", "-"), ("green", "-"), ("red", "-")],
        ylabel="GB/s",
        plot_name="softmax-performance",
        args={"M": 4096},
    )
)
def benchmark(M, N, provider):
    x = torch.randn(M, N, device=DEVICE, dtype=torch.float32)
    if provider == "torch":
        ms = triton.testing.do_bench(lambda: torch.softmax(x, axis=-1))
    if provider == "triton":
        ms = triton.testing.do_bench(lambda: softmax(x))
    if provider == "naive_softmax":
        ms = triton.testing.do_bench(lambda: naive_softmax(x))
    gbps = lambda ms: 2 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms)
